### Notebook to create BLOCK-T539 Initial Telescope Alignment

Created on: 2025-06-03

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os
import numpy as np

In [ ]:
current_path = os.getcwd()
block_number = 'T539'
name = "BLOCK-T539"
program = "BLOCK-T539"
reason = "initial_alignment"

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "day": {
        "description": "Day_obs for the reference state.",
        "type": "integer",
        "default": 1
    },
    "seq": {
        "description": "Sequence number for the reference state.",
        "type": "integer",
        "default": 1
    }
}

# Build the configuration schema
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

In [ ]:
setdof_script = ObservingScript(
    name="maintel/set_dof.py",
    standard=True,
    parameters= dict(
        day="$day",
        seq="$seq",
    )
)

closedloop_script_hexapod = ObservingScript(
    name="maintel/close_loop_lsstcam.py",
    standard=True,
    parameters= dict(
        program="$program",
        reason=reason,
        note="closed_loop_hexapods",
        exposure_time=30.0,
        max_iter=5,
        used_dofs=[0,1,2,3,4,5,6,7,8,9],
        truncation_index=6,
        gain_sequence=[0.75, 0.5, 0.5, 0.25]
    )
)

closedloop_script_alldofs = ObservingScript(
    name="maintel/close_loop_lsstcam.py",
    standard=True,
    parameters= dict(
        program="$program",
        reason=reason,
        note="closed_loop_alldofs",
        exposure_time=30.0,
        max_iter=5,
        used_dofs=np.arange(0,50).tolist(),
        truncation_index=11,
    )
)

scripts = []
scripts.append(setdof_script)
scripts.append(closedloop_script_hexapod)
scripts.append(closedloop_script_alldofs)

In [ ]:
block = ObservingBlock(
    name = program,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))